# 05 · Evaluación y Resultados Finales — SOL-USD
**Diplomado en Ciencia de Datos · ENES UNAM León**

Este notebook genera las visualizaciones y análisis finales para el **informe técnico** (sección 7. Resultados).

### Métricas utilizadas
| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MAE** | mean(\|y - ŷ\|) | Error promedio en USD — fácil de comunicar |
| **RMSE** | √mean((y-ŷ)²) | Penaliza errores grandes — más exigente |
| **MAPE** | mean(\|y-ŷ\|/y)×100 | Error en % — independiente de la escala |
| **R²** | 1 - SS_res/SS_tot | Cuánta varianza explica el modelo (0-1) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Cargar predicciones guardadas
df_preds = pd.read_csv('../data/processed/predicciones_test.csv', index_col=0, parse_dates=True)
print(df_preds.head())

## 5.1 Métricas finales (tabla resumen)

In [ ]:
def metricas(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    return {'MAE ($)': round(mae,2), 'RMSE ($)': round(rmse,2), 'MAPE (%)': round(mape,2), 'R²': round(r2,4)}

tabla = pd.DataFrame({
    'Ridge Regression': metricas(df_preds['real'], df_preds['pred_ridge']),
    'XGBoost':          metricas(df_preds['real'], df_preds['pred_xgb'])
}).T

print('\n===== MÉTRICAS FINALES — TEST SET =====')
print(tabla.to_string())
tabla

## 5.2 Gráfica principal: Predicción vs Real

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for ax, model, color, title in zip(
    axes,
    ['pred_ridge', 'pred_xgb'],
    ['#FFB800', '#14F195'],
    ['Ridge Regression', 'XGBoost']
):
    ax.plot(df_preds.index, df_preds['real'], label='Real', color='#9945FF', linewidth=2)
    ax.plot(df_preds.index, df_preds[model], label=f'Predicción {title}', color=color,
            linewidth=1.5, linestyle='--', alpha=0.9)
    
    # Banda de error ±MAE
    mae_val = mean_absolute_error(df_preds['real'], df_preds[model])
    ax.fill_between(df_preds.index,
                    df_preds[model] - mae_val,
                    df_preds[model] + mae_val,
                    alpha=0.15, color=color, label=f'Banda ±MAE (${mae_val:.1f})')
    ax.set_title(f'{title} — Predicción vs Real', fontsize=12)
    ax.set_ylabel('USD')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('../data/raw/05_prediccion_final.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Análisis de residuos

Un buen modelo tiene residuos centrados en 0, sin patrones y distribuidos normalmente.

In [ ]:
df_preds['residuo_ridge'] = df_preds['real'] - df_preds['pred_ridge']
df_preds['residuo_xgb']   = df_preds['real'] - df_preds['pred_xgb']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (model, color, nombre) in enumerate([
    ('residuo_ridge', '#FFB800', 'Ridge'),
    ('residuo_xgb',   '#14F195', 'XGBoost')
]):
    # Residuos en el tiempo
    axes[i, 0].plot(df_preds.index, df_preds[model], color=color, linewidth=1, alpha=0.8)
    axes[i, 0].axhline(0, color='red', linewidth=1, linestyle='--')
    axes[i, 0].set_title(f'{nombre} — Residuos en el tiempo')
    axes[i, 0].set_ylabel('Error (USD)')

    # Histograma de residuos
    axes[i, 1].hist(df_preds[model], bins=25, color=color, edgecolor='white', alpha=0.85)
    axes[i, 1].axvline(0, color='red', linewidth=1.5, linestyle='--')
    axes[i, 1].axvline(df_preds[model].mean(), color='white', linewidth=1.5,
                       label=f'Media: ${df_preds[model].mean():.2f}')
    axes[i, 1].set_title(f'{nombre} — Distribución de residuos')
    axes[i, 1].legend()

plt.tight_layout()
plt.savefig('../data/raw/05_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Scatter: Real vs Predicho

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, color, nombre in zip(
    axes,
    ['pred_ridge', 'pred_xgb'],
    ['#FFB800', '#14F195'],
    ['Ridge Regression', 'XGBoost']
):
    ax.scatter(df_preds['real'], df_preds[model], color=color, alpha=0.6, s=30)
    
    # Línea perfecta
    lims = [min(df_preds['real'].min(), df_preds[model].min()),
            max(df_preds['real'].max(), df_preds[model].max())]
    ax.plot(lims, lims, color='red', linewidth=1.5, linestyle='--', label='Predicción perfecta')
    
    r2 = r2_score(df_preds['real'], df_preds[model])
    ax.set_title(f'{nombre} | R² = {r2:.4f}')
    ax.set_xlabel('Real (USD)')
    ax.set_ylabel('Predicho (USD)')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/raw/05_scatter_real_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.5 Simulación de señales de trading

Una forma intuitiva de presentar los resultados en el video: ¿el modelo predice correctamente la dirección del movimiento?

In [ ]:
# Dirección real vs predicha
prev_close = df_preds['real'].shift(1)

real_dir = np.sign(df_preds['real'] - prev_close)          # +1 sube, -1 baja
pred_ridge_dir = np.sign(df_preds['pred_ridge'] - prev_close)
pred_xgb_dir   = np.sign(df_preds['pred_xgb']   - prev_close)

acc_ridge = (real_dir == pred_ridge_dir).mean() * 100
acc_xgb   = (real_dir == pred_xgb_dir).mean()   * 100

print('===== Precisión de dirección (¿sube o baja?) =====')
print(f'Ridge Regression : {acc_ridge:.1f}%')
print(f'XGBoost          : {acc_xgb:.1f}%')
print(f'\n(Random baseline: ~50%)')
print(f'Si el modelo supera 55% de dirección correcta en crypto, es relevante.')

## 5.6 Tabla resumen final para el informe

In [ ]:
tabla_final = pd.DataFrame({
    'Modelo': ['Ridge Regression (baseline)', 'XGBoost'],
    'MAE ($)':  [
        round(mean_absolute_error(df_preds['real'], df_preds['pred_ridge']), 2),
        round(mean_absolute_error(df_preds['real'], df_preds['pred_xgb']), 2)
    ],
    'RMSE ($)': [
        round(np.sqrt(mean_squared_error(df_preds['real'], df_preds['pred_ridge'])), 2),
        round(np.sqrt(mean_squared_error(df_preds['real'], df_preds['pred_xgb'])), 2)
    ],
    'MAPE (%)': [
        round(np.mean(np.abs((df_preds['real'] - df_preds['pred_ridge']) / df_preds['real'])) * 100, 2),
        round(np.mean(np.abs((df_preds['real'] - df_preds['pred_xgb'])   / df_preds['real'])) * 100, 2)
    ],
    'R²': [
        round(r2_score(df_preds['real'], df_preds['pred_ridge']), 4),
        round(r2_score(df_preds['real'], df_preds['pred_xgb']), 4)
    ],
    'Dir. Accuracy (%)': [round(acc_ridge, 1), round(acc_xgb, 1)]
})

print('\n' + '='*60)
print('TABLA FINAL — COPIA ESTA AL INFORME TÉCNICO (Sección 7)')
print('='*60)
print(tabla_final.to_string(index=False))

tabla_final.to_csv('../data/processed/tabla_resultados_finales.csv', index=False)
print('\nTabla guardada ✓')

---
## ✅ Proyecto completado

### Archivos generados
```
data/
  raw/
    sol_usd_raw.csv                  # Datos históricos crudos
    01_precio_volumen.png
    02_distribucion.png
    02_retornos_volatilidad.png
    02_correlacion.png
    02_acf_pacf.png
    02_mensual.png
    03_*.png                         # Visualizaciones de features
    04_*.png                         # Visualizaciones de modelos
    05_*.png                         # Evaluación final
  processed/
    sol_usd_eda.csv
    sol_usd_features.csv
    predicciones_test.csv
    ridge_model.pkl
    scaler.pkl
    xgboost_model.json
    tabla_resultados_finales.csv     ← Para el informe técnico
```

### Para el video (10-15 min)
1. **Problema** (2 min): ¿Por qué predecir Solana? Contexto de crypto
2. **Datos** (2 min): Mostrar la gráfica de precio y volumen
3. **Metodología** (3 min): Features creados, división temporal, modelos
4. **Resultados** (4 min): Predicción vs real, MAPE, dirección accuracy
5. **Conclusiones** (2 min): ¿Qué funciona? ¿Qué sigue? (LSTM, más features)

### Próximos pasos (si hay tiempo)
- Agregar LSTM para comparación con deep learning
- Walk-forward validation (más robusto para series temporales)
- Hyperparameter tuning con Optuna